# Lab: What is LLM Inference?

In this lab, you'll observe the inference process hands-on:
- Watch tokenization in action
- Measure prefill vs decode timing
- See the KV cache grow token by token
- Compare sampling strategies (greedy, top-k, top-p)
- Measure TTFT and inter-token latency

In [ ]:
import sys
sys.path.insert(0, '../../..')
import torch
import torch.nn as nn
import numpy as np
import time

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

## Stage 1: Tokenization

In [ ]:
# Demonstrate tokenization with a simple BPE-style tokenizer
# (In practice you'd use transformers.AutoTokenizer)
try:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained('meta-llama/Llama-3.1-8B')
except:
    # Fallback: simulate with GPT-2 tokenizer (publicly available)
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained('gpt2')

text = "What is the capital of France?"
tokens = tokenizer.encode(text)
print(f'Input: "{text}"')
print(f'Tokens: {tokens}')
print(f'Token count: {len(tokens)}')
print(f'\nToken breakdown:')
for t in tokens:
    print(f'  {t:6d} -> "{tokenizer.decode([t])}"')

## Stage 2: Prefill vs Decode Timing

In [ ]:
# Minimal transformer to demonstrate prefill vs decode difference
class MiniTransformer(nn.Module):
    def __init__(self, d_model=512, n_heads=8, n_layers=6):
        super().__init__()
        layer = nn.TransformerEncoderLayer(d_model, n_heads, dim_feedforward=2048, batch_first=True)
        self.layers = nn.TransformerEncoder(layer, n_layers)
        self.lm_head = nn.Linear(d_model, 32000)  # vocab
    
    def forward(self, x):
        return self.lm_head(self.layers(x))

model = MiniTransformer().to(device).eval()
print(f'Model params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')

In [ ]:
# Prefill: process many tokens at once
prompt_lengths = [32, 64, 128, 256, 512, 1024]
prefill_times = []

for seq_len in prompt_lengths:
    x = torch.randn(1, seq_len, 512, device=device)
    torch.cuda.synchronize() if device == 'cuda' else None
    
    start = time.perf_counter()
    with torch.no_grad():
        _ = model(x)
    torch.cuda.synchronize() if device == 'cuda' else None
    prefill_times.append(time.perf_counter() - start)

print('=== Prefill (process all tokens at once) ===')
print(f'{"Tokens":<10} {"Time (ms)":<12} {"Tokens/sec":<12}')
for length, t in zip(prompt_lengths, prefill_times):
    print(f'{length:<10} {t*1000:<12.1f} {length/t:<12.0f}')
print('\nNotice: throughput INCREASES with prompt length (GPU parallelism!)')

In [ ]:
# Decode: process one token at a time (simulating autoregressive generation)
n_tokens_to_generate = 50
decode_times = []

x = torch.randn(1, 1, 512, device=device)  # single token
torch.cuda.synchronize() if device == 'cuda' else None

for i in range(n_tokens_to_generate):
    start = time.perf_counter()
    with torch.no_grad():
        _ = model(x)
    torch.cuda.synchronize() if device == 'cuda' else None
    decode_times.append(time.perf_counter() - start)

print('=== Decode (one token at a time) ===')
print(f'Average time per token: {np.mean(decode_times)*1000:.1f} ms')
print(f'Throughput: {1/np.mean(decode_times):.0f} tokens/sec')
print(f'\nCompare: prefill processed 1024 tokens in {prefill_times[-1]*1000:.0f}ms')
print(f'         decode would need {n_tokens_to_generate} × {np.mean(decode_times)*1000:.1f}ms = {sum(decode_times)*1000:.0f}ms for just {n_tokens_to_generate} tokens')
print(f'\n→ Decode is {(np.mean(decode_times)*1024)/(prefill_times[-1]):.0f}× slower per token than prefill')

## Stage 3: KV Cache Growth

In [ ]:
# Simulate KV cache memory growth
# Llama 3.1 8B: 32 layers, 8 KV heads, 128 dim, FP16
n_layers = 32
n_kv_heads = 8
head_dim = 128
bytes_per_param = 2  # FP16

kv_per_token = 2 * n_layers * n_kv_heads * head_dim * bytes_per_param  # K + V
print(f'KV cache per token: {kv_per_token:,} bytes = {kv_per_token/1024:.0f} KB')
print()

print(f'{"Context length":<20} {"KV cache size":<15} {"Notes"}')
print('-' * 60)
for ctx in [128, 512, 1024, 4096, 8192, 32768, 131072]:
    size_mb = ctx * kv_per_token / 1e6
    size_gb = size_mb / 1000
    note = ''
    if size_mb > 1000:
        note = f'⚠️ {size_gb:.1f} GB — exceeds A10G!'
    elif size_mb > 500:
        note = '⚡ Half an A10G'
    print(f'{ctx:<20} {size_mb:<15.0f} MB {note}')

print(f'\n→ At 128K context, KV cache alone = {131072 * kv_per_token / 1e9:.1f} GB')
print(f'  That\'s MORE than the 16 GB model itself!')

## Stage 4: Sampling Strategies

In [ ]:
# Demonstrate different sampling strategies
torch.manual_seed(42)

# Simulate logits (model output before softmax)
vocab_size = 10
token_names = ['the', 'a', 'Paris', 'France', 'is', 'was', 'capital', 'city', 'of', '.']
logits = torch.tensor([2.5, 1.0, 5.2, 0.5, 3.1, 0.8, 1.5, 0.3, 2.0, 0.1])

probs = torch.softmax(logits, dim=-1)
print('Token probabilities (after softmax):')
for name, p in sorted(zip(token_names, probs.tolist()), key=lambda x: -x[1]):
    print(f'  {name:<10} {p:.3f} {"█" * int(p * 50)}')

print('\n--- Greedy (always pick highest) ---')
print(f'  Selected: "{token_names[logits.argmax()]}" (p={probs[logits.argmax()]:.3f})')

print('\n--- Top-k (k=3, sample from top 3) ---')
topk = torch.topk(probs, k=3)
print(f'  Candidates: {[token_names[i] for i in topk.indices.tolist()]}')
print(f'  Probs (renormalized): {(topk.values / topk.values.sum()).tolist()}')

print('\n--- Temperature (T=0.5 vs T=2.0) ---')
probs_cold = torch.softmax(logits / 0.5, dim=-1)
probs_hot = torch.softmax(logits / 2.0, dim=-1)
print(f'  T=0.5 (confident): top token gets {probs_cold.max():.3f}')
print(f'  T=1.0 (normal):    top token gets {probs.max():.3f}')
print(f'  T=2.0 (creative):  top token gets {probs_hot.max():.3f}')

## Stage 5: End-to-End Metrics (TTFT, ITL)

In [ ]:
# Simulate a complete inference request and measure key metrics
prompt_len = 256
gen_tokens = 30

# Prefill
x = torch.randn(1, prompt_len, 512, device=device)
torch.cuda.synchronize() if device == 'cuda' else None
t0 = time.perf_counter()
with torch.no_grad():
    _ = model(x)
torch.cuda.synchronize() if device == 'cuda' else None
ttft = time.perf_counter() - t0

# Decode
itls = []
x_tok = torch.randn(1, 1, 512, device=device)
for _ in range(gen_tokens):
    torch.cuda.synchronize() if device == 'cuda' else None
    start = time.perf_counter()
    with torch.no_grad():
        _ = model(x_tok)
    torch.cuda.synchronize() if device == 'cuda' else None
    itls.append(time.perf_counter() - start)

total_latency = ttft + sum(itls)

print('=== Inference Metrics ===')
print(f'Prompt: {prompt_len} tokens | Generated: {gen_tokens} tokens')
print(f'')
print(f'TTFT (Time to First Token):  {ttft*1000:.1f} ms')
print(f'ITL (Inter-Token Latency):   {np.mean(itls)*1000:.1f} ms (avg), {np.percentile(itls, 95)*1000:.1f} ms (p95)')
print(f'Total Latency:               {total_latency*1000:.0f} ms')
print(f'Throughput:                   {gen_tokens/sum(itls):.0f} tokens/sec (decode only)')
print(f'')
print(f'Time breakdown:')
print(f'  Prefill: {ttft/total_latency*100:.0f}% ({prompt_len} tokens in {ttft*1000:.0f}ms)')
print(f'  Decode:  {sum(itls)/total_latency*100:.0f}% ({gen_tokens} tokens in {sum(itls)*1000:.0f}ms)')

## Key Takeaways

| Observation | Implication |
|-------------|-------------|
| Prefill throughput increases with prompt length | GPU parallelism — more tokens = better utilization |
| Decode is ~Nx slower per token than prefill | Memory-bound — reads entire model for 1 token |
| KV cache grows linearly with context | 128 KB/token × context = memory pressure |
| Sampling strategy affects output quality | Temperature/top-k are the user-facing knobs |
| TTFT depends on prompt length | Long prompts = slow first response |

**Next:** Module 0.2 — Why these observations make LLM inference fundamentally different from traditional ML.